# Analise de Topicos em Emails

**Status:** Em andamento · **Área:** topicos-emails

Projeto coletivo de especialização: pré-processamento, busca de tópicos por coerência e LDA.

Este arquivo está identificado como **Em andamento**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


# Projeto de especialização Senac.

* Robert de Jesus Araújo
* Elivanio Geraldo
* Daniel Figueiredo

# Importações

In [ ]:
!pip install -q --break-system-packages \
    pyLDAvis==3.4.1 \
    gensim==4.3.3 \
    spacy==3.7.2 \
    pandas==2.2.2 \
    scikit-learn==1.5.0

!python -m spacy download pt_core_news_sm -q

In [ ]:
import pandas as pd
import re, string, itertools

import spacy, nltk, gensim
import pyLDAvis
from pyLDAvis import gensim_models as gensimvis

from nltk.corpus import stopwords
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

nltk.download("stopwords", quiet=True)

import unidecode


# Desenvolvimento

In [ ]:
# Dados
df = pd.read_csv("/content/dataset_emails.csv")
texts_raw = (df["assunto"] + " " + df["corpo"]).astype(str).tolist()


In [ ]:
data = df
topico = "Solicitações de avaliação de projetos digitais"
print(f"Foram obtidas {data.shape[0]} {topico}")
data.head()

In [ ]:
# Pré-processamento
nlp = spacy.load("pt_core_news_sm")
stop_pt = set(stopwords.words("portuguese"))
my_stop = stop_pt | {"att", "re", "fw"}

def clean_doc(doc):
    doc = unidecode.unidecode(doc)#acentos
    doc = ''.join([char for char in doc if not char.isdigit()])#remove numeros
    doc = doc.lower()#minusculas
    doc = doc.translate(str.maketrans("", "", string.punctuation))#pontuações
    doc_sp = nlp(doc)
    doc = ' '.join(doc.split())#espaços

    kept = [
        tok.lemma_ for tok in doc_sp
        if tok.pos_ in {'NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV'} and tok.lemma_ not in my_stop
    ]

    return kept
    """
    Limpeza de texto por classes gramaticais utilizando spaCy.
    :param text: Texto a ser limpo.
    :param allowed_postags: Lista de tags POS permitidas.
    :return: Texto limpo com apenas as classes gramaticais permitidas.
    """
texts_tok = [clean_doc(t) for t in texts_raw]

In [ ]:
# Dicionário & corpus
dictionary = Dictionary(texts_tok)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in texts_tok]


In [ ]:
# Busca de hiper-parâmetros
def compute_coherence(k, alpha, eta):
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        alpha=alpha,
        eta=eta,
        passes=10,
        iterations=200,
        random_state=42,
    )
    cm = CoherenceModel(model=lda, texts=texts_tok,
                        dictionary=dictionary, coherence='c_v')
    return lda, cm.get_coherence()

results = []
for k in range(3, 7):
    lda, coh = compute_coherence(k, 'symmetric', 'auto')
    results.append((k, coh))
best_k = max(results, key=lambda x: x[1])[0]
print("Melhor K:", best_k)

In [ ]:
# Treino final (ajuste alpha/beta se quiser)
lda_final = LdaModel(
    corpus, id2word=dictionary,
    num_topics=best_k, alpha='symmetric', eta='auto',
    passes=20, iterations=400, random_state=42
)

In [ ]:
# Visualização
import pyLDAvis
vis = pyLDAvis.gensim_models.prepare(lda_final, corpus, dictionary)
pyLDAvis.display(vis)


In [ ]:
rotulos = {
    0: "Solicitações de avaliação de projetos digitais",
    1: "Cronogramas de projetos internos",
    2: "Lançamentos e materiais promocionais",
    3: "Marcos / entregas de sistemas",
    4: "Convocações de RH / reuniões operacionais",
}

for idx, topic in lda_final.show_topics(num_topics=-1, formatted=False):
    palavras = ", ".join([w for w, _ in topic[:5]])
    nome     = rotulos.get(idx, " Revisar rótulo")
    print(f"Tópico {idx}: {palavras:<60} = {nome}")


In [ ]:
# Matrizes doc-tópico (opcional)
df_topics = pd.DataFrame(
    [dict(lda_final[bow]) for bow in corpus]
).fillna(0)


## Continuidade

Preservar três autores; acrescentar import/dependência unidecode; padronizar ambiente e unificar com três variantes LDA; usar amostra sintética/anonimizada e revisar saídas para não expor e-mails. Relacionar rótulos aos tópicos efetivamente treinados.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.